In [1]:
import numpy as np


In [2]:
def p_adic_norm(n, p=2):
    if n == 0:
        return 0
    count = 0
    while n % p == 0:
        n //= p
        count += 1
    return p**(-count)


In [3]:
w_target = 2**10 + 2**9 + 2**0
w_current = 0

rng = np.random.default_rng(seed=42)

for i in range(1_000):
    shift = 2**rng.integers(0, 20)

    loss_before = p_adic_norm(w_current - w_target)

    w_new = w_current + shift

    loss_after = p_adic_norm(w_new - w_target)

    if loss_after < loss_before:
        w_current = w_new
        print(f"Step {i}: Loss {loss_after}, Weights {w_current}")

    if loss_after == 0:
        print("Reached target")
        break


Step 55: Loss 0.001953125, Weights 1
Step 69: Loss 0.0009765625, Weights 513
Step 114: Loss 0, Weights 1537
Reached target


We observe that the model first fixed the "root" at 1, then refined the second digit to 512, and finally arrived at a leaf equal to the target.

In other words, optimization in p-adic space operates not as a "smooth creeping" toward the target, but as the sequential fixing of digits.

Let's try a larger target and the selection range.

In [4]:
w_target = 2**25 + 2**15 + 2**11 + 2**8 + 2**6 + 2**5 + 2**4 + 2**2 + 2**1
w_current = 0

rng = np.random.default_rng(seed=42)

for i in range(100_000):
    shift = 2**rng.integers(0, 1556)

    loss_before = p_adic_norm(w_current - w_target)

    w_new = w_current + shift

    loss_after = p_adic_norm(w_new - w_target)

    if loss_after < loss_before:
        w_current = w_new
        print(f"Step {i}: Loss {loss_after}, Weights {w_current}")

    if loss_after == 0:
        print("Reached target")
        break


Step 1487: Loss 0.25, Weights 2
Step 1948: Loss 0.0625, Weights 6
Step 3190: Loss 0.03125, Weights 22
Step 6413: Loss 0.015625, Weights 54
Step 6520: Loss 0.00390625, Weights 118
Step 8870: Loss 0.00048828125, Weights 374
Step 10061: Loss 3.0517578125e-05, Weights 2422
Step 12024: Loss 2.9802322387695312e-08, Weights 35190
Step 13237: Loss 0, Weights 33589622
Reached target


/tmp/ipykernel_6885/2956296303.py:13: RuntimeWarning: overflow encountered in scalar subtract
  loss_after = p_adic_norm(w_new - w_target)


And yet, even with such enormous values, we converge quite rapidly. This, once again, confirms that with each iteration, the model refines increasingly distant digits.

And yet, let's further expand the target and the selection range.

In [5]:
w_target = 2**35 + 2**25 + 2**15 + 2**11 + 2**8 + 2**6 + 2**5 + 2**4 + 2**2 + 2**1
w_current = 0

rng = np.random.default_rng(seed=42)

for i in range(300_000):
    shift = 2**rng.integers(0, 15556)

    loss_before = p_adic_norm(w_current - w_target)

    w_new = w_current + shift

    loss_after = p_adic_norm(w_new - w_target)

    if loss_after < loss_before:
        w_current = w_new
        print(f"Step {i}: Loss {loss_after}, Weights {w_current}")

    if loss_after == 0:
        print("Reached target")
        break


/tmp/ipykernel_6885/1760053785.py:13: RuntimeWarning: overflow encountered in scalar subtract
  loss_after = p_adic_norm(w_new - w_target)


Step 24981: Loss 0.25, Weights 2
Step 46337: Loss 0.0625, Weights 6
Step 47031: Loss 0.03125, Weights 22
Step 84680: Loss 0.015625, Weights 54
Step 95755: Loss 0.00390625, Weights 118
Step 98706: Loss 0.00048828125, Weights 374
Step 109529: Loss 3.0517578125e-05, Weights 2422
Step 113009: Loss 2.9802322387695312e-08, Weights 35190
Step 162826: Loss 2.9103830456733704e-11, Weights 33589622
Step 221704: Loss 0, Weights 34393327990
Reached target


With standard training, we would never reach the end; however, due to the regularization properties of p-adic numbers, the model itself focuses solely on refining increasingly distant digits.

Yet, up to this point, we have been incrementing only a single digit at a time, but what if we were to increment several at once?

In [6]:
w_target = 2**35 + 2**25 + 2**15 + 2**11 + 2**8 + 2**6 + 2**5 + 2**4 + 2**2 + 2**1
w_current = 0

rng = np.random.default_rng(seed=42)

for i in range(300_000):
    loss_before = p_adic_norm(w_current - w_target)

    ww = 0
    for j in range(rng.integers(0, 25)):
        shift = 2**rng.integers(0, 36)
        ww += shift

    w_new = w_current + ww

    loss_after = p_adic_norm(w_new - w_target)

    if loss_after < loss_before:
        w_current = w_new
        print(f"Step {i}: Loss {loss_after}, Weights {w_current}")

    if loss_after == 0:
        print("Reached target")
        break


Step 5: Loss 0.0625, Weights 22067032774
Step 13: Loss 0.03125, Weights 56704938710
Step 20: Loss 0.0078125, Weights 65311945462
Step 25: Loss 0.001953125, Weights 100745493366
Step 275: Loss 0.00048828125, Weights 144249373046
Step 390: Loss 3.0517578125e-05, Weights 144254962038
Step 854: Loss 7.62939453125e-06, Weights 144322234742
Step 944: Loss 3.814697265625e-06, Weights 147610962294
Step 1399: Loss 9.5367431640625e-07, Weights 147611224438
Step 1609: Loss 1.1920928955078125e-07, Weights 164794239350
Step 1647: Loss 5.960464477539063e-08, Weights 164802627958
Step 1766: Loss 2.9802322387695312e-08, Weights 164819405174
Step 3076: Loss 1.862645149230957e-09, Weights 199212697974
Step 3679: Loss 4.656612873077393e-10, Weights 199749568886
Step 4406: Loss 2.3283064365386963e-10, Weights 201897052534
Step 5387: Loss 2.9103830456733704e-11, Weights 206192019830
Step 5510: Loss 1.4551915228366852e-11, Weights 240551758198
Step 67900: Loss 3.637978807091713e-12, Weights 309271234934


Wow, in just 5510 steps, we achieved a result that, using bitwise modification, took 221704 steps to reach. At the same time, however, our weights exploded; and while the loss became very close to zero, it was not exactly zero.

Let's try limiting the weights, for instance, to $2^{64}$.

In [7]:
w_target = 34_359_738_368 + 33_554_432 + 32768 + 2048 + 256 + 64 + 32 + 16 + 4 + 2
w_current = 0

rng = np.random.default_rng(seed=42)

for i in range(300_000):
    loss_before = p_adic_norm(w_current - w_target)

    ww = 0
    for j in range(rng.integers(0, 25)):
        shift = 2**rng.integers(0, 36)
        ww += shift

    w_new = (w_current + ww) % (2**64)

    loss_after = p_adic_norm(w_new - w_target)

    if loss_after < loss_before:
        w_current = w_new
        print(f"Step {i}: Loss {loss_after}, Weights {w_current}")

    if loss_after == 0:
        print("Reached target")
        break


OverflowError: Python int too large to convert to C long